# 10.1 Semantic meaning with embeddings

Explore pretrained word and document embeddings. Add your own code and markdown cells after each prompt and before its checks. Preserve supplied cells and IDs. All 100 points are awarded manually: 70 code and 30 interpretation. Assertions check required results, not the quality of semantic conclusions. Supplied plotting and clustering code is not scored. Read the README.


In [ ]:
from pathlib import Path
from tempfile import gettempdir
import os
import csv
import gc

download_root = Path(gettempdir()) / "compint" / "10.1.embeddings"
for name in ("gensim", "sentence-transformers", "huggingface"):
    (download_root / name).mkdir(parents=True, exist_ok=True)
os.environ["GENSIM_DATA_DIR"] = str(download_root / "gensim")
os.environ["HF_HOME"] = str(download_root / "huggingface")
os.environ["HF_HUB_CACHE"] = str(download_root / "huggingface" / "hub")
# Configure directories BEFORE importing downloaders. Restart if already imported.
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gensim.downloader as api
from sentence_transformers import SentenceTransformer, util
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

activity_dir = Path.cwd() if (Path.cwd() / "data/documents.csv").exists() else Path.cwd().parent
with (activity_dir / "data/documents.csv").open(encoding="utf-8", newline="") as file:
    records = list(csv.DictReader(file))
document_ids = [r["id"] for r in records]
documents = [r["text"] for r in records]
print("Downloaded assets:", download_root)
for doc_id, text in zip(document_ids, documents):
    print(doc_id, text)


Temporary caches may be removed by your operating system. First execution downloads models; later executions reuse cached files. 


In [ ]:
word_labels = ["king", "queen", "man", "woman", "dog", "cat", "horse", "bird", "car", "bus", "train", "bicycle", "paris", "france", "japan", "tokyo", "banana", "apple", "student", "teacher"]
glove_words = word_labels.copy()
capitalized = {"paris": "Paris", "france": "France", "japan": "Japan", "tokyo": "Tokyo"}
word2vec_words = [capitalized.get(word, word) for word in word_labels]
word_pairs = [("king", "queen"), ("king", "banana"), ("car", "bus"), ("student", "teacher")]


## 1. Word embeddings (40 points)

### A. Create embeddings (20 points)

Define **`embed(words)`**, taking a list of vocabulary tokens and returning a NumPy matrix with one embedding row per input word, in the same order. This function reads from the currently selected pretrained **`model`**; it does not train anything. Conceptually, `embed(words)` is the word-level counterpart of SentenceTransformer's `encode(documents)`.

Load `glove_model` with `api.load("glove-wiki-gigaword-100")`. Select it with `model = glove_model`, then call your function to create **`glove_embeddings = embed(glove_words)`**. Load `word2vec_model` with `api.load("word2vec-google-news-300")`, select it with `model = word2vec_model`, and create **`word2vec_embeddings = embed(word2vec_words)`**. Define the function once and reuse it; do not store these matrices in a dictionary. Every supplied token is in its corresponding model's vocabulary; preserve capitalization.

For each model, print its name, vocabulary size, vector dimension, matrix shape, and the shape of one word vector. Do not print raw embeddings. Hints: `model[token]`, `np.stack`, `vector_size`, `len(model)`. Keep both model objects available for B and C; the supplied cleanup cell releases them afterward.

Add your function, model loading, and embedding calls in a new code cell below.


In [ ]:
assert "embed" in globals() and callable(embed), "Define embed(words)."
assert "glove_embeddings" in globals() and "word2vec_embeddings" in globals()
for embeddings, dimension in ((glove_embeddings, 100), (word2vec_embeddings, 300)):
    assert isinstance(embeddings, np.ndarray)
    assert embeddings.shape == (len(word_labels), dimension)
    assert np.isfinite(embeddings).all()
# Check ordering and repeated words with each selected model.
model = glove_model
assert np.allclose(embed([glove_words[1], glove_words[0], glove_words[1]]), glove_embeddings[[1, 0, 1]])
model = word2vec_model
assert np.allclose(embed([word2vec_words[1], word2vec_words[0], word2vec_words[1]]), word2vec_embeddings[[1, 0, 1]])


### B. Similarities and neighbors (10 points)

Using the model objects from A, create `glove_scores` and `word2vec_scores`: four cosine similarities in `word_pairs` order. Use `glove_model.similarity` and `word2vec_model.similarity`. Create `glove_neighbors` and `word2vec_neighbors` containing the five nearest neighbors of `student`; print their words and scores. Hint: `most_similar(..., topn=5)`.

The supplied heatmap displays your pair scores. Add this analysis in a new code cell below, separate from your embedding code.


In [ ]:
assert "glove_scores" in globals() and "word2vec_scores" in globals()
for scores in (glove_scores, word2vec_scores):
    values = np.asarray(scores)
    assert values.shape == (len(word_pairs),) and np.isfinite(values).all()
    assert (np.abs(values) <= 1.00001).all()
assert len(glove_neighbors) == len(word2vec_neighbors) == 5


In [ ]:
def show_similarity_heatmap(scores, row_labels, column_labels, title):
    # Illustration only: the supplied scores were computed by the student.
    scores = np.asarray(scores)
    fig, ax = plt.subplots(figsize=(max(6, len(column_labels) * 0.55), max(4, len(row_labels) * 0.45)))
    sns.heatmap(scores, xticklabels=column_labels, yticklabels=row_labels,
                annot=True, fmt=".2f", cmap="viridis", vmin=-1, vmax=1,
                annot_kws={"size": 8}, cbar_kws={"label": "Cosine similarity"}, ax=ax)
    ax.set(title=title)
    ax.tick_params(axis="x", labelrotation=45)
    ax.tick_params(axis="y", labelrotation=0)
    fig.tight_layout()
    plt.show()

show_similarity_heatmap(
    np.column_stack([glove_scores, word2vec_scores]),
    [f"{a} / {b}" for a, b in word_pairs],
    ["GloVe", "Word2Vec"], "Word-pair cosine similarities")


### C. Analogies (10 points)

In a separate new code cell, evaluate **king − man + woman** and **Paris − France + Japan** in both models, respecting capitalization. **Choose at least three more analogies** and test the same expressions in both models (at least five per model). Print every expression, intended relationship, and five `(word, score)` candidates. Check vocabulary membership and select other words if needed; unexpected answers are valid findings.

Store results in `glove_analogies` and `word2vec_analogies`, dictionaries with descriptive analogy names as keys and five candidates as values. Include `royalty` and `capitals`, plus at least three additional keys, and use matching keys across both dictionaries. Hint: `most_similar(positive=[...], negative=[...], topn=5)`.


In [ ]:
assert "glove_analogies" in globals() and "word2vec_analogies" in globals()
for results in (glove_analogies, word2vec_analogies):
    assert {"royalty", "capitals"}.issubset(results)
    assert len(results) >= 5, "Try at least three additional analogies in each model."
    assert all(len(candidates) == 5 for candidates in results.values())
assert set(glove_analogies) == set(word2vec_analogies)


In [ ]:
# Keep the small embedding matrices and results for plots; release full models.
del model, glove_model, word2vec_model
gc.collect()


## 2. Word visualization

PCA projects learned features into two coordinates that preserve some variance, not all semantic relationships. It does not assign names to dimensions. Each model gets a separate projection; their axes and raw vectors are not interchangeable.


In [ ]:
def plot_embeddings(embeddings, labels, title):
    pca = PCA(n_components=2, svd_solver="full")
    coordinates = pca.fit_transform(np.asarray(embeddings))
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.scatter(coordinates[:, 0], coordinates[:, 1])
    for label, point in zip(labels, coordinates):
        ax.annotate(label, point, xytext=(4, 4), textcoords="offset points")
    ax.set(xlabel="Principal component 1", ylabel="Principal component 2",
           title=f"{title} — variance retained: {pca.explained_variance_ratio_.sum():.1%}")
    ax.set_aspect("equal", adjustable="datalim")
    plt.show()
    return coordinates

plot_embeddings(glove_embeddings, glove_words, "GloVe")
plot_embeddings(word2vec_embeddings, word2vec_words, "Word2Vec")


**Interpretation Q2 (5 points)** Analyze where each word is projected into the 2D plot. Do you notice anything in common between words that are closer together?

Add a markdown answer below.

## 3. Embed full documents (20 points)

Create `sentence_model` using `SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu", cache_folder=str(download_root / "sentence-transformers"))`. Use its `encode` method to create `document_embeddings` for `documents`, retaining input order and returning a NumPy array. Print model name, matrix shape, and the shape of one document vector with its document ID and text. Do not print raw embedding values.

This pretrained model produces 384 features per text. It is intended for sentences and short paragraphs; inputs exceeding its default 256 wordpiece limit are truncated. The supplied texts are short. Do not apply the earlier count/TF-IDF preprocessing: preserve natural text. No model training or averaging of static word vectors is requested.


In [ ]:
assert "document_embeddings" in globals(), "Create document_embeddings."
assert np.asarray(document_embeddings).shape == (len(documents), 384)
assert np.isfinite(document_embeddings).all()
assert (np.linalg.norm(document_embeddings, axis=1) > 0).all()


## 4. Compare documents — code: 10 points

Create `document_similarities` using `util.cos_sim` on the original `document_embeddings` against themselves. The result is a tensor; use `.detach().cpu().numpy()` to obtain `similarity_array` for display and sorting. The supplied Seaborn heatmap displays the matrix with document labels and annotated scores; you do not need to implement visualization. Brighter colors indicate higher similarity (dark purple is lower, yellow is higher) on the shared −1 to 1 scale. The diagonal shows self-similarity, so inspect off-diagonal cells to compare different documents.

Print scores and full texts for pairs D00/D01, D00/D06, D06/D18, and D00/D05. Create `nearest_document_ids` as the three integer row indices most similar to row 0, excluding row 0 itself. Display their document IDs, scores, and texts. Hint: sort the unrounded row in descending order.

Compute cosine in the original 384-dimensional space, before PCA. Scores range from −1 to 1 and are not probabilities. Similarity can reflect shared subject matter without agreement or logical equivalence.


In [ ]:
assert "document_similarities" in globals() and "similarity_array" in globals()
assert similarity_array.shape == (len(documents), len(documents))
assert np.isfinite(similarity_array).all()
assert np.allclose(similarity_array, similarity_array.T, atol=1e-5)
assert np.allclose(np.diag(similarity_array), 1, atol=1e-5)
assert (np.abs(similarity_array) <= 1.00001).all()
assert "nearest_document_ids" in globals()
assert len(nearest_document_ids) == len(set(nearest_document_ids)) == 3
assert all(isinstance(i, (int, np.integer)) and 0 < i < len(documents) for i in nearest_document_ids)
expected_ids = np.argsort(-similarity_array[0], kind="stable")
assert list(nearest_document_ids) == [int(i) for i in expected_ids if i != 0][:3]


In [ ]:
show_similarity_heatmap(similarity_array, document_ids, document_ids, "Document cosine similarities")


**Interpretation Q4 (10 points)** Use the supplied heatmap to identify a high-similarity pair of different documents and a low-similarity pair. Cite their IDs and scores, as well as scores for the four specified pairs. Do paraphrases remain similar despite different words? Does shared `bank` vocabulary imply shared meaning? Does a high score imply agreement in the negation pair? Interpret the three retrieved documents and one surprising result. Add a markdown answer.


## 5. Visualize documents

IDs connect points to the complete texts printed above. Keep these exact coordinates for every clustering comparison.


In [ ]:
document_xy = plot_embeddings(document_embeddings, document_ids, "Document embeddings")


**Interpretation Q5 (5 points)** How many "groupings" do you identify? Once you identify groupings, select at least 2 documents from each and read them fully. What do they have in common? How does the groupings relate to the similarity scores? You can add as many cells as you need to display the documents.

Add a markdown answer below.

## 6. Experiment with clustering

KMeans is a clustering algorithm that asks you to supply `k`; it partitions the 2D points into that many groups. All experiments here operate on the same 2D projection, not the original embedding space. Cluster IDs/colors do not correspond across runs.

Use the complete helper below in a **new code cell**, choosing at least three distinct `k` values between 2 and 23. For example, call `compare_clusters(document_xy, k_values=(2, 4, 6))`; choose and record your own values. Keep the corpus and coordinates fixed. You may make additional calls. No clustering implementation is required or scored.


In [ ]:
def compare_clusters(points, k_values=(2, 4, 6)):
    k_values = tuple(k_values)
    if len(set(k_values)) < 3 or any(not isinstance(k, int) or not 2 <= k < len(points) for k in k_values):
        raise ValueError("Choose at least three distinct integer k values from 2 to n_documents - 1.")
    results = {f"KMeans k={k}": KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(points)
               for k in k_values}
    fig, axes = plt.subplots(1, len(results), figsize=(6 * len(results), 6), sharex=True, sharey=True)
    for ax, (name, labels) in zip(axes, results.items()):
        groups = np.unique(labels)
        ax.scatter(points[:, 0], points[:, 1], c=labels, cmap="tab20")
        for doc_id, point in zip(document_ids, points):
            ax.annotate(doc_id, point, xytext=(3, 3), textcoords="offset points")
        ax.set(title=f"{name}: {len(groups)} clusters", xlabel="PC1", ylabel="PC2")
        ax.set_aspect("equal", adjustable="box")
        print(f"\n{name}: {len(groups)} clusters")
        for group in groups:
            indices = np.flatnonzero(labels == group)
            print(f"  Cluster {group}: {len(indices)} documents")
            for i in indices:
                print(f"    {document_ids[i]}: {documents[i]}")
    plt.show()
    return results


**Interpretation Q6 (10 points)** Record at least three distinct `k` values. Compare which documents merge or split as `k` changes, citing document IDs and explaining how the chosen number of groups affects the partition. Propose names for the groups in each result and support your interpretations with specific document memberships, including mixed or unclear groups. Which grouping is more useful for this corpus, and why? Discuss how does this clustering compares with your visual inspection in the section prior to this, as well as with the similarity scores.

Add your comparison call and markdown answer below. No numeric quality threshold or expected cluster membership is required.
